# 3. Molecular Dynamics Trajectory Analysis

This notebook contains scripts for analyzing molecular dynamics trajectories of FRD-chalcone complexes (13e, 14c, 14l).

## 3.1 Trajectory Alignment

The first step is to align all MD trajectories using `cpptraj`. This process includes:
- Removal of water and ions
- Protein centering and alignment
- Generation of clean trajectories in NetCDF format
- Creation of dry topologies and reference structures without solvent

In [1]:
%%writefile trajectory_alignment.sh

#!/bin/bash

mkdir -p ./md_trajs

for system in 13e 14c 14l; do

    for replica in {1..5}; do

        cpptraj << EOF
        
        # Load the topology
        parm ./chalcone-${system}/FRD-${system}.parm7

        # Load the trajectory 1-100 ns
        trajin ./chalcone-${system}/md${replica}/md?.nc
        trajin ./chalcone-${system}/md${replica}/md??.nc
        trajin ./chalcone-${system}/md${replica}/md???.nc

        # Load the reference structure 
        reference ./chalcone-${system}/FRD-${system}.rst7

        # Align the trajectory to the reference structure using the specified residues and atom types
        strip :WAT,Cl-
        center :1-1147 mass origin
        image origin center byres familiar
        rms reference :392-398,415-418,422-425,427-431,545-547,589-591,616-620,624-625,628,662,805-806,841-843,852-861@CA,C,N,O

        # Save the aligned trajectory in NetCDF format
        trajout ./md_trajs/${system}_rep${replica}.nc netcdf
        go
        quit
EOF
    done

    cpptraj << EOF
        parm ./chalcone-${system}/FRD-${system}.parm7
        parmstrip :WAT,Cl-
        parmwrite ./md_trajs/FRD-${system}_dry.parm7
        go

        clear all

        parm ./chalcone-${system}/FRD-${system}.parm7
        trajin ./chalcone-${system}/FRD-${system}.rst7
        strip :WAT,Cl-
        trajout ./md_trajs/FRD-${system}_dry.rst7
        go
EOF

done

Writing trajectory_alignment.sh


In [ ]:
!bash trajectory_alignment.sh

## 3.2 RMSD and Distance Calculations

Structural stability analysis through:
- **Ligand RMSD**: Deviation of the UNL ligand relative to the reference structure (excluding hydrogen atoms)
- **Protein RMSD**: Deviation of the backbone (CA, C, N, O) of residues 1-1147
- **Key distance**: Distance between the chalcone alpha carbon (C1) and Cys862 sulfur atom (SG)

In [2]:
%%writefile md_analysis_rmsd_and_distance.sh

#!/bin/bash

mkdir -p ./md_analysis/rmsd_distance

for system in 13e 14c 14l; do

    for replica in {1..5}; do

        cpptraj << EOF
        
        # Load the topology
        parm ./md_trajs/FRD-${system}_dry.parm7

        # Load the aligned trajectory
        trajin ./md_trajs/${system}_rep${replica}.nc

        # Load the reference structure 
        reference ./md_trajs/FRD-${system}_dry.rst7

        # Calculate RMSD of the ligand
        rms nofit reference ":UNL&!@H" out ./md_analysis/rmsd_distance/${system}_rep${replica}_ligand_rmsd.dat

        # Calculate RMSD of the protein backbone
        rms reference ":1-1147@CA,C,N,O" out ./md_analysis/rmsd_distance/${system}_rep${replica}_rmsd.dat

        # Calculate distance between alpha carbon of chalcone and sulfur atom of Cys862
        distance dist$replica ":UNL@C1" ":862@SG" out ./md_analysis/rmsd_distance/${system}_rep${replica}_dist.dat
        go
        quit
EOF
    done
done

      

Writing md_analysis_rmsd_and_distance.sh


In [ ]:
bash md_analysis_rmsd_and_distance.sh


CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools)
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| Date/time: 04/22/26 23:02:24
| Available memory: 12.403 GB

INPUT: Reading input from 'STDIN'
  [parm ./md_trajs/FRD-13e_dry.parm7]
	Reading './md_trajs/FRD-13e_dry.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin ./md_trajs/13e_rep1.nc]
	Reading './md_trajs/13e_rep1.nc' as Amber NetCDF
  [reference ./md_trajs/FRD-13e_dry.rst7]
	Reading './md_trajs/FRD-13e_dry.rst7' as Amber Restart
	Setting active reference for distance-based masks: 'FRD-13e_dry.rst7'
  [rms nofit reference ":UNL&!@H" out ./md_analysis/rmsd_distance/13e_rep1_ligand_rmsd.dat]
    RMSD: (:UNL&!@H), reference is "Cpptraj Generated Restart" (:UNL&!@H).
	No fitting will be performed.
  [rms reference ":1-1147@CA,C,N,O" out ./md_analysis/rmsd_distance/13e_rep1_rmsd.dat]
    RMSD: (:1-1147@CA,C,N,O), reference is "Cpptraj Generated Restart" (:1-1147@CA,C,N,O).
	Best-fit RMS

## 3.3 Hydrogen Bond Analysis

Identification and quantification of hydrogen bonds formed between the ligand and protein during the simulation. This analysis reveals key interactions for complex stability.

In [4]:
%%writefile md_analysis_hbonds.sh

#!/bin/bash

mkdir -p ./md_analysis/hbonds

for system in 13e 14c 14l; do

    cpptraj <<EOF
    # Load topology and trajectories
    parm ./md_trajs/FRD-${system}_dry.parm7
    trajin ./md_trajs/${system}_rep1.nc
    trajin ./md_trajs/${system}_rep2.nc
    trajin ./md_trajs/${system}_rep3.nc
    trajin ./md_trajs/${system}_rep4.nc
    trajin ./md_trajs/${system}_rep5.nc
    
    # Load reference structure
    reference ./md_trajs/FRD-${system}_dry.rst7

    # Calculate hydrogen bonds between protein and ligand
    hbond contacts :1-1147,UNL dist 3.5 avgout ./md_analysis/hbonds/${system}_avg.dat series uuseries ./md_analysis/hbonds/${system}_hbond_series.dat nointramol
    go
    
    # Create time series of number of hydrogen bonds
    create ./md_analysis/hbonds/${system}_nhbvtime.dat contacts[UU]
    go
    quit
EOF

done

Writing md_analysis_hbonds.sh


In [ ]:
bash md_analysis_hbonds.sh


CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools)
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| Date/time: 04/22/26 23:17:24
| Available memory: 12.357 GB

INPUT: Reading input from 'STDIN'
  [parm ./md_trajs/FRD-13e_dry.parm7]
	Reading './md_trajs/FRD-13e_dry.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin ./md_trajs/13e_rep1.nc]
	Reading './md_trajs/13e_rep1.nc' as Amber NetCDF
  [trajin ./md_trajs/13e_rep2.nc]
	Reading './md_trajs/13e_rep2.nc' as Amber NetCDF
  [trajin ./md_trajs/13e_rep3.nc]
	Reading './md_trajs/13e_rep3.nc' as Amber NetCDF
  [trajin ./md_trajs/13e_rep4.nc]
	Reading './md_trajs/13e_rep4.nc' as Amber NetCDF
  [trajin ./md_trajs/13e_rep5.nc]
	Reading './md_trajs/13e_rep5.nc' as Amber NetCDF
  [reference ./md_trajs/FRD-13e_dry.rst7]
	Reading './md_trajs/FRD-13e_dry.rst7' as Amber Restart
	Setting active reference for distance-based masks: 'FRD-13e_dry.rst7'
  [hbond contacts :1-1147,UNL dist 3.5 avgout ./md_an

### 3.3.1 Process Hydrogen Bond Data

Statistical analysis of hydrogen bonds for each system. The script will:
- Process each system (13e, 14c, 14l) individually
- Calculate mean and standard deviation for each hydrogen bond interaction
- Save individual system results
- Consolidate all systems into a single file for comparison

In [3]:
import pandas as pd
import numpy as np
import os

systems = ['13e', '14c', '14l']
all_results = []

for system in systems:
    print(f"\n{'='*50}")
    print(f"Processing system: {system}")
    print('='*50)
    
    # Read the hbond series file for this system
    file_path = f'./md_analysis/hbonds/{system}_hbond_series.dat'
    
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found, skipping...")
        continue
    
    data = pd.read_csv(file_path, sep='\s+')
    
    # Get headers
    header = data.columns.tolist()
    
    # Calculate average and standard deviation for each column
    averages = data.mean()
    std_devs = data.std()
    
    # Create results DataFrame for this system
    results = pd.DataFrame({
        'System': system,
        'Header': header[1:],  # Exclude first column 'Frame Number'
        'Average': averages[1:],
        'Standard_Deviation': std_devs[1:]
    })
    
    # Save individual system results
    output_file = f'./md_analysis/hbonds/{system}_processed_hbonds.csv'
    results.to_csv(output_file, index=False)
    
    # Add to consolidated results
    all_results.append(results)
    
    print(f"System {system}: {len(header)-1} hydrogen bond interactions detected")
    print(f"Results saved to: {output_file}")

# Consolidate all systems into one file
if all_results:
    consolidated = pd.concat(all_results, ignore_index=True)
    consolidated_file = './md_analysis/hbonds/all_systems_hbonds.csv'
    consolidated.to_csv(consolidated_file, index=False)
    print(f"\n{'='*50}")
    print(f"Consolidated results saved to: {consolidated_file}")
    print(f"Total interactions across all systems: {len(consolidated)}")
    
    # Post-process consolidated file
    !sed -i 's/-/,/g' ./md_analysis/hbonds/all_systems_hbonds.csv
    !sed -i 's/_/,/g' ./md_analysis/hbonds/all_systems_hbonds.csv
    !sed -i 's/@/,/g' ./md_analysis/hbonds/all_systems_hbonds.csv
    
    # Extract ligand interactions
    !grep UNL ./md_analysis/hbonds/all_systems_hbonds.csv > ./md_analysis/hbonds/all_systems_hbonds_ligand.csv || echo "System,Header,Average,Standard_Deviation" > ./md_analysis/hbonds/all_systems_hbonds_ligand.csv
    
    print("Ligand-specific interactions extracted to: all_systems_hbonds_ligand.csv")

<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_47258/1603584132.py:20: SyntaxWarning: invalid escape sequence '\s'
  data = pd.read_csv(file_path, sep='\s+')



Processing system: 13e
System 13e: 10 hydrogen bond interactions detected
Results saved to: ./md_analysis/hbonds/13e_processed_hbonds.csv

Processing system: 14c
System 14c: 30 hydrogen bond interactions detected
Results saved to: ./md_analysis/hbonds/14c_processed_hbonds.csv

Processing system: 14l
System 14l: 64 hydrogen bond interactions detected
Results saved to: ./md_analysis/hbonds/14l_processed_hbonds.csv

Consolidated results saved to: ./md_analysis/hbonds/all_systems_hbonds.csv
Total interactions across all systems: 104
Ligand-specific interactions extracted to: all_systems_hbonds_ligand.csv


## 3.4 Native Contacts Analysis

Evaluation of close contacts (< 5.5 Å) between the ligand and protein residues. This analysis identifies residues that interact most frequently with the ligand during the simulation.

In [1]:
%%writefile md_analysis_contacts.sh

#!/bin/bash

mkdir -p ./md_analysis/contacts

for system in 13e 14c 14l; do

    cd ./md_analysis/contacts
    mkdir -p ${system}
    cd ${system}

    cpptraj <<EOF
    # Load topology and trajectories
    parm ../../../md_trajs/FRD-${system}_dry.parm7
    trajin ../../../md_trajs/${system}_rep1.nc
    trajin ../../../md_trajs/${system}_rep2.nc
    trajin ../../../md_trajs/${system}_rep3.nc
    trajin ../../../md_trajs/${system}_rep4.nc
    trajin ../../../md_trajs/${system}_rep5.nc
    
    # Load reference structure
    reference ../../../md_trajs/FRD-${system}_dry.rst7

    # Calculate native contacts between ligand (UNL) and protein residues
    nativecontacts name ${system}_contacts :UNL&!@H= :1-1147&!@H= byresidue out ${system}_contacts_res.dat noimage mindist maxdist distance 5.5 resout ${system}_contact_pairs.dat series seriesout ${system}_native_series.dat savenonnative nncontactpdb ${system}_nonnative.pdb seriesnnout ${system}_nonnative_series.dat writecontacts ${system}_contacts.out contactpdb ${system}_native.pdb resseries present resseriesout ${system}_resseries.dat
    go
    quit
EOF

    cd ../../..

done

Writing md_analysis_contacts.sh


In [ ]:
bash md_analysis_contacts.sh


CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools)
    ___  ___  ___  ___
     | \/ | \/ | \/ | 
    _|_/\_|_/\_|_/\_|_

| Date/time: 04/22/26 23:59:31
| Available memory: 11.502 GB

INPUT: Reading input from 'STDIN'
  [parm ../../../md_trajs/FRD-13e_dry.parm7]
	Reading '../../../md_trajs/FRD-13e_dry.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin ../../../md_trajs/13e_rep1.nc]
	Reading '../../../md_trajs/13e_rep1.nc' as Amber NetCDF
  [trajin ../../../md_trajs/13e_rep2.nc]
	Reading '../../../md_trajs/13e_rep2.nc' as Amber NetCDF
  [trajin ../../../md_trajs/13e_rep3.nc]
	Reading '../../../md_trajs/13e_rep3.nc' as Amber NetCDF
  [trajin ../../../md_trajs/13e_rep4.nc]
	Reading '../../../md_trajs/13e_rep4.nc' as Amber NetCDF
  [trajin ../../../md_trajs/13e_rep5.nc]
	Reading '../../../md_trajs/13e_rep5.nc' as Amber NetCDF
  [reference ../../../md_trajs/FRD-13e_dry.rst7]
	Reading '../../../md_trajs/FRD-13e_dry.rst7' as Amber Restart
	Setting active reference 

In [4]:
import pandas as pd
import numpy as np
import os

systems = ['13e', '14c', '14l']
all_results = []

for system in systems:
    print(f"\n{'='*50}")
    print(f"Processing system: {system}")
    print('='*50)
    
    # Read the contact resseries file for this system
    file_path = f'./md_analysis/contacts/{system}/{system}_resseries.dat'
    
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found, skipping...")
        continue
    
    data = pd.read_csv(file_path, sep='\s+')
    
    # Get headers
    header = data.columns.tolist()
    
    # Calculate average and standard deviation for each column
    averages = data.mean()
    std_devs = data.std()
    
    # Create results DataFrame for this system
    results = pd.DataFrame({
        'System': system,
        'Header': header[1:],  # Exclude first column 'Frame Number'
        'Average': averages[1:],
        'Standard_Deviation': std_devs[1:]
    })
    
    # Save individual system results
    output_file = f'./md_analysis/contacts/{system}/{system}_processed_contacts.csv'
    results.to_csv(output_file, index=False)
    
    # Add to consolidated results
    all_results.append(results)
    
    print(f"System {system}: {len(header)-1} contact interactions detected")
    print(f"Results saved to: {output_file}")

# Consolidate all systems into one file
if all_results:
    consolidated = pd.concat(all_results, ignore_index=True)
    consolidated_file = './md_analysis/contacts/all_systems_contacts.csv'
    consolidated.to_csv(consolidated_file, index=False)
    print(f"\n{'='*50}")
    print(f"Consolidated results saved to: {consolidated_file}")
    print(f"Total interactions across all systems: {len(consolidated)}")
    
    # Post-process consolidated file
    !sed -i 's/nn_//g' ./md_analysis/contacts/all_systems_contacts.csv
    !sed -i 's/_/,/g' ./md_analysis/contacts/all_systems_contacts.csv
    !sed -i 's/:/,/g' ./md_analysis/contacts/all_systems_contacts.csv
    
    print("Post-processing completed for consolidated contacts file")

<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_47258/2039165169.py:20: SyntaxWarning: invalid escape sequence '\s'
  data = pd.read_csv(file_path, sep='\s+')



Processing system: 13e
System 13e: 85 contact interactions detected
Results saved to: ./md_analysis/contacts/13e/13e_processed_contacts.csv

Processing system: 14c
System 14c: 99 contact interactions detected
Results saved to: ./md_analysis/contacts/14c/14c_processed_contacts.csv

Processing system: 14l
System 14l: 105 contact interactions detected
Results saved to: ./md_analysis/contacts/14l/14l_processed_contacts.csv

Consolidated results saved to: ./md_analysis/contacts/all_systems_contacts.csv
Total interactions across all systems: 289
Post-processing completed for consolidated contacts file


## 3.5 Free Energy Calculation (MM-PBSA)

Estimation of binding free energy using the MM-PBSA method (Molecular Mechanics Poisson-Boltzmann Surface Area). This analysis provides quantitative information about protein-ligand binding affinity and per-residue energy contributions.

In [ ]:
%%writefile md_analysis_mmpbsa.sh

#!/bin/bash

set -e

mkdir -p ./md_analysis/mmpbsa

for system in 13e 14c 14l; do

    echo "Processing system: ${system}"
    
    # Create output directories
    mkdir -p ./md_analysis/mmpbsa/${system}/trajins

    # Step 1: Process trajectories and prepare files for MM-PBSA
    cpptraj << EOF
    # Load topology and reference structure
    parm ./chalcone-${system}/FRD-${system}.parm7
    trajin ./chalcone-${system}/FRD-${system}.rst7
    
    # Remove ions and water
    strip :Na+,Cl-
    strip :WAT@EPW
    
    # Save clean PDB for structure preparation
    trajout ./md_analysis/mmpbsa/${system}/trajins/complex_mmpbsa.pdb
    go

    clear parm
    clear trajin

    # Load topology and all MD trajectories
    parm ./chalcone-${system}/FRD-${system}.parm7
    trajin ./chalcone-${system}/md1/md?.nc 1 last 50
    trajin ./chalcone-${system}/md1/md??.nc 1 last 50
    trajin ./chalcone-${system}/md1/md???.nc 1 last 50
    trajin ./chalcone-${system}/md2/md?.nc 1 last 50
    trajin ./chalcone-${system}/md2/md??.nc 1 last 50
    trajin ./chalcone-${system}/md2/md???.nc 1 last 50
    trajin ./chalcone-${system}/md3/md?.nc 1 last 50
    trajin ./chalcone-${system}/md3/md??.nc 1 last 50
    trajin ./chalcone-${system}/md3/md???.nc 1 last 50
    trajin ./chalcone-${system}/md4/md?.nc 1 last 50
    trajin ./chalcone-${system}/md4/md??.nc 1 last 50
    trajin ./chalcone-${system}/md4/md???.nc 1 last 50
    trajin ./chalcone-${system}/md5/md?.nc 1 last 50
    trajin ./chalcone-${system}/md5/md??.nc 1 last 50
    trajin ./chalcone-${system}/md5/md???.nc 1 last 50
    reference ./chalcone-${system}/FRD-${system}.rst7

    # Process trajectories: remove solvent, center, and align
    strip :Na+,Cl-
    strip :WAT@EPW
    center :1-1147
    image origin center
    autoimage
    rms reference :392-398,415-418,422-425,427-431,545-547,589-591,616-620,624-625,628,662,805-806,841-843,852-861@CA,C,N,O

    # Save processed trajectory
    trajout ./md_analysis/mmpbsa/${system}/trajins/trajin_mmpbsa.nc netcdf
    go

    clear parm
    clear trajin
    clear reference

    # Create dry topology (no ions, no water)
    parm ./chalcone-${system}/FRD-${system}.parm7
    parmstrip :Na+,Cl-
    parmstrip :WAT@EPW
    parmwrite out ./md_analysis/mmpbsa/${system}/trajins/parm_mmpbsa.parm7
    go
    quit
EOF

    # Step 2: Prepare PDB with pdb4amber
    pdb4amber -i ./md_analysis/mmpbsa/${system}/trajins/complex_mmpbsa.pdb -o ./md_analysis/mmpbsa/${system}/trajins/complex_mmpbsa_amber.pdb

    # Step 3: Add PDB information to topology using parmed
    parmed <<EOF
    parm ./md_analysis/mmpbsa/${system}/trajins/parm_mmpbsa.parm7
    addPDB ./md_analysis/mmpbsa/${system}/trajins/complex_mmpbsa_amber.pdb
    outparm ./md_analysis/mmpbsa/${system}/trajins/parm_mmpbsa_parmed.parm7
    quit
EOF

    # Step 4: Create MM-PBSA input file
    cat << EOF > ./md_analysis/mmpbsa/${system}/mmpbsa.in
&general
  startframe=1, endframe=100, interval=10, verbose=3, keep_files=0,
  entropy=0, netcdf=1,
/
&pb
  istrng=150.0, inp=2, sasopt=3, ipb=2, radiopt=1, indi=5.0,
/
&decomp
  idecomp=3,
  print_res="1-1147"
  dec_verbose=3,
/
EOF

    # Step 5: Prepare input topologies for MM-PBSA
    cd ./md_analysis/mmpbsa/${system}
    
    ante-MMPBSA.py -p ./trajins/parm_mmpbsa_parmed.parm7 -c ./trajins/complex.parm7 -r ./trajins/prot.parm7 -l ./trajins/lig.parm7 -s ':Na+,Cl-,WAT' -n ':UNL' --radii=mbondi2

    # Step 6: Run MM-PBSA calculation
    mpirun -np 10 --use-hwthread-cpus MMPBSA.py.MPI -O -i mmpbsa.in \
        -o final_results_MMPBSA.dat -do final_decomposition_MMPBSA.dat \
        -sp ./trajins/parm_mmpbsa_parmed.parm7 -cp ./trajins/complex.parm7 \
        -rp ./trajins/prot.parm7 -lp ./trajins/lig.parm7 -eo energy_frame.csv \
        -y ./trajins/trajin_mmpbsa.nc

    cd ../../..
    
    echo "System ${system} completed"
    
done

echo "MM-PBSA analysis completed for all systems"

In [ ]:
bash md_analysis_mmpbsa.sh